In [4]:
# SUMMARY ------------------------
import math
import numpy as np
import matplotlib.pyplot as plt
import random
import torch
import torch.nn.functional as F
%matplotlib inline

In [5]:
# read in all the words
words = open('names.txt','r').read().splitlines()
words[:8]

['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']

In [6]:
#build the vocabulary of characters and mappings to/from integers
chars = sorted(list(set(''.join(words))))
stoi = { s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
vocab_size = len(itos)
print(itos)
print(vocab_size)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}
27


In [7]:
#build the dataset
block_size = 3 #context length:how many characters do we take to predict the next one?
def build_dataset(words):
    X,Y = [],[]
    for w in words:
        context = [0] * block_size
        for ch in w + '.':
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix]
    X = torch.tensor(X)
    Y = torch.tensor(Y)
    print(X.shape,Y.shape)
    return X,Y

import random
random.seed(42)
random.shuffle(words)
n1 = int(0.8*len(words))
n2 = int(0.9*len(words))

Xtr,Ytr = build_dataset(words[:n1])
Xdev,Ydev = build_dataset(words[n1:n2])
Xte,Yte = build_dataset(words[n2:])

torch.Size([182625, 3]) torch.Size([182625])
torch.Size([22655, 3]) torch.Size([22655])
torch.Size([22866, 3]) torch.Size([22866])


In [8]:
# let's train a deeper network

class Linear:
    def __init__(self,fan_in,fan_out,bias=True):
        self.weight = torch.randn((fan_in,fan_out),generator=g)/fan_in ** 0.5
        self.bias = torch.zeros(fan_out) if bias else None

    def __call__(self,x):
        self.out = x @ self.weight
        if self.bias is not None:
            self.out += self.bias
        return self.out

    def parameters(self):
        return [self.weight]+([] if self.bias is None else [self.bias])

class BatchNorm1d:
    def __init__(self,dim,eps=1e-5,momentum=0.1):
        self.eps = eps
        self.momentum = momentum
        self.training = True
        #parameters (train with backdrop)
        self.gamma  = torch.ones(dim)
        self.beta = torch.zeros(dim)
        #buffer(trained with a running 'momentum update')
        self.running_mean = torch.zeros(dim)
        self.running_var = torch.ones(dim)

        def __call__(self,x):
            # calculate the forward pass
            if self.training:
                xmean = x.mean(0,keepdim = True) # batch mean
                xvar = x.var(0,keepdim = True,unbiased = True) #batch variance
            else:
                xmean = self.running_mean
                xvar = self.running_var
            xhat = (x-xmean)/torch.sqrt(xvar +self.eps) # normalize to unit variance
            self.out = self.gamma * xhat + self.beta
            #update the buffers
            if self.training:
                with torch.no_grad():
                    self.running_mean = (1-self.momentum)*self.running_mean +self.momentum * xmean
                    self.running_var = (1-self.momentum)*self.running_var +self.momentum * xvar
            return self.out

        def parameters(self):
            return [self.gamma,self.beta]


class Tanh:
    def __call__(self,x):
        self.out = torch.tanh(x)
        return self.out
    def parameters(self):
        return []

n_embd = 10 #the dimensionality of the character embedding vectors
n_hidden = 100 #the number of neurons in the hidden of the MLP
g = torch.Generator().manual_seed(2147483647) # for reproducibility

C = torch.randn((vocab_size,n_embd), generator = g)
layers = [
    Linear(n_embd * block_size,n_hidden),Tanh(),
    Linear(n_hidden,n_hidden),Tanh(),
    Linear(n_hidden,n_hidden),Tanh(),
    Linear(n_hidden,n_hidden),Tanh(),
    Linear(n_hidden,n_hidden),Tanh(),
    Linear(n_hidden,vocab_size),
]

with torch.no_grad():
    #last layer:make less confident
    layers[-1].weight *= 0.1
    #all other layers:apply gain
    for layer in layers[:-1]:
        if isinstance(layer,Linear):
            layer.weight *= 5/3

parameters = [C] + [p for layer  in layers for p in layer.parameters()]
print(sum(p.nelement() for p in parameters))
for p in parameters:
    p.requires_grad =True

46497


In [9]:
#same optimization as last time
max_steps = 200000
batch_size = 32
lossi = []
ud = []

for i in range(max_steps):

    #minibatch construct
    ix = torch.randint(0,Xtr.shape[0],(batch_size,),generator = g)
    Xb,Yb = Xtr[ix],Ytr[ix] #batch X,Y

    #forward pass
    emb = C[Xb] #embed the characters into vectors
    x = emb.view(emb.shape[0],-1) #concatenate the vectors
    for layer in layers:
        x = layer(x)
    loss = F.cross_entropy(x,Yb) # loss function

    #backward pass
    for layer in layers:
        layer.out.retain_grad() #AFTER_DEBUG；would take out retain_grape
    for p in parameters:
        p.grad = None
    loss.backward()

    #update
    lr = 0.1 if i < 100000 else 0.01 #step learning rate decay
    for p in parameters:
        p.data += -lr * p.grad

    #track stats
    if i%10000 == 0 :
        print(f'{i:7d}/{max_steps:7d}:{loss.item():.4f}')
    lossi.append(loss.log10().item())
    with torch.no_grad():
        ud.append([(lr*p.grad.std() / p.data.std()).log().item() for p in parameters])
    i +=1
    if i > 100000:
        break #AFTER_DEBUG；would take out obviously to run full optimization
        

      0/ 200000:3.2962


KeyboardInterrupt: 

In [ ]:
# visualize histogtams
plt.figure(figsize =(20,4)) #width and height of the plot
legends = []
for i,layer in enumerate(layers[:-1]): #notes;exclude the output layer
    if isinstance(layer,Tanh):
        t = layer.out
        print('layer %d(%10s):mean %+.2f,std %.2f,saturated:%.2f%%'%(i,layer.__class__.__name__,t.mean(),t.std(),(t.abs()>0.97).float().mean()*100))
        hy,hx = torch.histogram(t,density = True)
        plt.plot(hx[:-1].detach(),hy.detach())
        legends.append(f'layer{i} ({layer.__class__.__name__})')

plt.legend(legends)
plt.title('activation distribution')

In [ ]:
# visualize histogtams
plt.figure(figsize =(20,4)) #width and height of the plot
legends = []
for i,layer in enumerate(layers[:-1]): #notes;exclude the output layer
    if isinstance(layer,Tanh):
        t = layer.out.grad
        print('layer %d(%10s):mean %+.2f,std %.2f,saturated:%.2f%%'%(i,layer.__class__.__name__,t.mean(),t.std(),(t.abs()>0.97).float().mean()*100))
        hy,hx = torch.histogram(t,density = True)
        plt.plot(hx[:-1].detach(),hy.detach())
        legends.append(f'layer{i} ({layer.__class__.__name__})')

plt.legend(legends)
plt.title('gradient distribution')

In [ ]:
# visualize histogtams
plt.figure(figsize =(20,4)) #width and height of the plot
legends = []
for i,p in enumerate(parameters): #notes;exclude the output layer
    t = p.grad
    print('weight %10s| mean %+f|std %e,grad:data ratio:%e'%(tuple(p.shape),t.mean(),t.std(),t.std()/p.std()))
    hy,hx = torch.histogram(t,density = True)
    plt.plot(hx[:-1].detach(),hy.detach())
    legends.append(f'layer{i} ({tuple(p.shape)})')

plt.legend(legends)
plt.title('weights gradient distribution')

In [ ]:
plt.figure(figsize=(20,4))
legends = []
for i,p in enumerate(parameters):
    if p.ndim == 2:
        plt.plot([ud[j][i] for j in range(len(ud))])
        legends.append('param %d'%i)
plt.plot([0,len(ud)],[-3,-3],'k') #these ratios should be -1~-3,indicate on plot ,数值越小说明训练越缓慢
plt.legend(legends)